In [5]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers>=4.41.0 trl==0.4.7

In [6]:
!pip install bitsandbytes


In [5]:
!pip install triton==2.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 6.7 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires triton==3.2.0; platform_system == "Linux" and platform_machine == "x86_64", but you have triton 2.3.0 which is incompatible.


In [8]:
!pip install transformers

In [1]:
# !pip install -U bitsandbytes
import torch
print(torch.cuda.is_available())   # should be True
print(torch.version.cuda)          # PyTorch CUDA version
print(torch.cuda.get_device_name(0))  # GPU name


True
12.8
Tesla T4


In [15]:
!pip uninstall -y transformers
!pip install transformers --upgrade


Found existing installation: transformers 4.55.2
Uninstalling transformers-4.55.2:
  Successfully uninstalled transformers-4.55.2
  Using cached transformers-4.55.2-py3-none-any.whl.metadata (41 kB)
Using cached transformers-4.55.2-py3-none-any.whl (11.3 MB)


In [ ]:
# importing necessary libraries

import os
import torch
from datasets import load_dataset
from transformers import pipeline
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    logging,
)
from peft import LoraConfig, PeftModel # freezing most of the weigths in training llms lets uses some weights
from trl import SFTTrainer


# `Dataset` https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k

# `Model` https://huggingface.co/NousResearch/Llama-2-7b-chat-hf

##Dataset for fine tuning

###full fine tuning : we need parameter efficient fine tuning techniques like LORA or QLORA

# step 3

1. load llama-2 chathf model (chat model)
2. train it with the data set for target specific tasks

qlora will use a rank of 64 with a scaling parametr of 16 well load the llama2 model directly in 4 bit precision using the
NF4 type and train it for one epoch

In [5]:
# model want to train from the hugging face hub
model_name = "NousResearch/Llama-2-7b-chat-hf"  # 70 b parameters

# the instaruction dataset to use
dataset_name ="mlabonne/guanaco-llama2-1k"

# fine tuned model
new_model = "llama-2-7b-chat-finetune"

#-------------------------------------
# qlora parameters

# lora attention dimension
lora_r = 64

# alpha paramerter for lora scaling
lora_alpha =16

# dropout probability for lora layers
lora_dropout = 0.1

#--------------------------------------
# bits and bytes parameters

# activate 4 bit precision base model loading
use_4bit = True

# compute dtype for 4-bit base model
bnb_4bit_compute_dtype = "float16"

#quantization type(fp4 or nf4)
bnb_4bit_quant_type = "nf4"


#activate nested quantuzed model group size for 4-bit base model(double quantization)
use_nested_quant = False


# --------------------------------------
# training arguments parameter

output_dir="./results"

num_train_epochs = 1

# enable fp16/bf16 training (set bf16 true with an a100)
fp16 = False
bf16 = False

# batch size per gpu for training
per_device_train_batch_size = 4

# gradient accumulation steps : num of update steps to accumulate gradients
gradient_accumulation_steps = 1

# enable gradient checkpointing
gradient_checkpointing = True

# maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# initial learning rate (adam optimizer)
learning_rate = 2e-4

# weight decay to apply to all layers except bias/layernorm weights
weight_decay = 0.001

# optimizer to use
optim = "paged_adamw_32bit"

# learning rate schedule
lr_scheduler_type = "cosine"

# num of training steps
max_steps =-1

# ratio of steps for a linear warmup
warmup_ratio = 0.03

# group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# save checkpoint every x updates steps
save_steps = 0

# log every x updates steps
logging_steps = 25

#---------------------------------------------------------
# sft parameters
max_seq = None

packing = False

# load model in gpu 0
device_map = {"": 0}

 ### Load everything and start fine tuning process



*   load data set we defined reformat the prompt filter
*   configure bitsand bytes for 4 bit quantization
*   loading the llama2 model in 4 bit precision on a gpu with corresponding tokens
*   we re loading configurations for qlora regular training parameters and passing the training can finally start



In [3]:
# !pip install -q -U bitsandbytes
# !pip install -U bitsandbytes
# !pip install -U bitsandbytes-windows
!pip install --force-reinstall --no-cache-dir bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 242.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 315.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 179.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 272.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 277.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 387.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 160.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 258.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 294.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
!pip install accelerate

In [ ]:
# load dataset
dataset = load_dataset(dataset_name, split="train")

# load tokenizer and model with alora configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

#  check gpu compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# load base model
'''
AutoModelForCausalLM is a generic loader for causal language models (LLMs used for text generation, like GPT, LLaMA, Mistral).

It figures out the right architecture (e.g., GPT2LMHeadModel, LlamaForCausalLM, etc.) based on the model_name you pass in.

“Causal” means it predicts the next token given all previous ones (unlike encoder-decoder models).

What from_pretrained does

Downloads the model weights and configuration from the Hugging Face Hub or a local path.

Initializes the model in Python with those weights.

Can also load with optimizations like quantization and device mapping.
'''
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp=1

#load llama tokenizer
# Tokenization in NLP - GeeksforGeeksA tokenizer is a component in Natural Language Processing (NLP) that breaks down text into smaller units called tokens

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # fix weird overflow issue with fp16 training

#load lora configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)
# set training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)


# set supervised fine tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# train model
trainer.train()

In [ ]:
# save the trained model
trainer.model.save_pretrained(new_model)

In [ ]:
# step 5 check the plots on tensor board
%load_ext tensorboard
%tensorboard --logdir results/runs

In [ ]:


# run text generation pipeline with our next model

prompt ="what is large language model"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])